In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from LegendreKANLayer import LegendreKANLayer
import random

# 设置设备为 GPU，如果可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 设置随机种子
seed = 55
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# 定义 Monge-Ampère 方程的源项和解析解（根据需求修改）
al = 3 / 4  # alpha 值

def source_function(x, y):
    r2 = x**2 + y**2
    return 4 * al**2 * (2 * al - 1) * r2**(2 * al - 2)

def analytical_solution(x, y):
    r2 = x**2 + y**2
    return r2**al

def analytical_gradient(x, y, al=3/4, eps=1e-12):
    """
    计算解析解 u = (x² + y²)^(3/4) 的梯度
    处理原点处的奇点问题
    
    参数:
    x, y: 输入张量
    al: 指数参数 (默认为3/4)
    eps: 避免除以零的小量
    
    返回:
    grad_x, grad_y: 对x和y的偏导数
    """
    # 计算半径平方
    r_sq = x**2 + y**2
    
    # 标识原点附近的点
    near_origin = r_sq < eps
    
    # 计算幂项 (al-1 = -1/4，需要避免除以零)
    safe_r_sq = torch.where(near_origin, eps, r_sq)
    power_term = al * torch.pow(safe_r_sq, al - 1)
    
    # 原点处设为零
    power_term = torch.where(near_origin, torch.zeros_like(power_term), power_term)
    
    # 计算梯度分量
    grad_x = power_term * (2 * x)
    grad_y = power_term * (2 * y)
    
    return grad_x, grad_y

# 定义求解模型
class LegenKAN(nn.Module):
    def __init__(self):
        super(LegenKAN, self).__init__()
        self.legenkan1 = LegendreKANLayer(2, 8, 6)
        self.legenkan2 = LegendreKANLayer(8, 8, 6)
        self.legenkan3 = LegendreKANLayer(8, 1, 6)

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)  # x: [N,1], y: [N,1] -> xy: [N,2]
        xy = self.legenkan1(xy)
        xy = self.legenkan2(xy)
        xy = self.legenkan3(xy)
        return xy

# 将模型放到 GPU
solver = LegenKAN().to(device)

# 定义"L"形区域的网格
# 上左矩形: x ∈ [-1,0], y ∈ [0,1]
x_values_tl = torch.linspace(-1, 0, 100).to(device)
y_values_tl = torch.linspace(0, 1, 100).to(device)
X_tl, Y_tl = torch.meshgrid(x_values_tl, y_values_tl, indexing="ij")
X_tl = X_tl.reshape(-1,1)
Y_tl = Y_tl.reshape(-1,1)

# 右下矩形: x ∈ [0,1], y ∈ [-1,0]
x_values_br = torch.linspace(0, 1, 100).to(device)
y_values_br = torch.linspace(-1, 0, 100).to(device)
X_br, Y_br = torch.meshgrid(x_values_br, y_values_br, indexing="ij")
X_br = X_br.reshape(-1,1)
Y_br = Y_br.reshape(-1,1)

# 合并两个矩形区域形成L形域
X = torch.cat([X_tl, X_br], dim=0)
Y = torch.cat([Y_tl, Y_br], dim=0)

# 定义边界条件掩码
boundary_mask_tl = (X_tl == -1) | (X_tl == 0) | (Y_tl == 0) | (Y_tl == 1)
boundary_mask_br = (X_br == 0) | (X_br == 1) | (Y_br == -1) | (Y_br == 0)
boundary_mask = torch.cat([boundary_mask_tl, boundary_mask_br], dim=0).squeeze()

interior_mask = ~boundary_mask

X_boundary, Y_boundary = X[boundary_mask], Y[boundary_mask]
X_interior, Y_interior = X[interior_mask], Y[interior_mask]

# 为内部点启用梯度
X_interior.requires_grad = True
Y_interior.requires_grad = True

# 损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(solver.parameters(), lr=0.01)

# 自适应采样参数
num_adaptive_steps = 5
num_high_error_samples = 200
learning_rate_decay = 0.8
adaptive_sample_increment = 20

loss_LK = []

# 训练
epochs = 10000
alpha = 0.0001
previous_loss = float('inf')

for epoch in range(epochs):
    optimizer.zero_grad()

    # 边界损失
    numerical_boundary = solver(X_boundary, Y_boundary)
    boundary_target = analytical_solution(X_boundary, Y_boundary)
    boundary_loss = criterion(numerical_boundary, boundary_target)

    # 内部点的 Monge-Ampère 方程损失
    numerical_interior = solver(X_interior, Y_interior)
    u_x = torch.autograd.grad(numerical_interior, X_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_y = torch.autograd.grad(numerical_interior, Y_interior, grad_outputs=torch.ones_like(numerical_interior), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, X_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, Y_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_xy = torch.autograd.grad(u_x, Y_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    hessian_det = u_xx * u_yy - u_xy**2
    interior_loss = criterion(hessian_det, source_function(X_interior, Y_interior))

    # 总损失
    loss = boundary_loss + alpha * interior_loss

    # 反向传播与优化
    loss.backward(retain_graph=True)
    optimizer.step()

    loss_LK.append(loss.item())

    # 学习率调整
    if loss.item() > previous_loss:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= learning_rate_decay
        print(f"Epoch [{epoch+1}/{epochs}], Loss increased. Reducing learning rate to: {param_group['lr']:.6f}")
    previous_loss = loss.item()

    # 每200个epoch输出信息
    if (epoch + 1) % 200 == 0:
        # 确保输入需要梯度
        X.requires_grad_(True)
        Y.requires_grad_(True)
        
        # 重置梯度标志
        numerical_solution_global = solver(X, Y)
        analytical_solution_global = analytical_solution(X, Y)
        
        # 1. 计算函数值误差
        error_global = torch.abs(numerical_solution_global - analytical_solution_global)
        max_error = torch.max(error_global).item()
        mean_error = torch.mean(error_global).item()
        l2_error_sq = torch.mean((numerical_solution_global - analytical_solution_global)**2)
        l2_error = torch.sqrt(l2_error_sq).item()
        
        try:
            # 2. 计算数值解梯度
            grad_numerical_x, grad_numerical_y = torch.autograd.grad(
                outputs=numerical_solution_global,
                inputs=[X, Y],
                grad_outputs=torch.ones_like(numerical_solution_global),
                create_graph=False,
                retain_graph=True
            )
            
            # 3. 计算解析解梯度 (使用已知公式而非自动微分)
            grad_analytical_x, grad_analytical_y = analytical_gradient(X, Y)  # 推荐使用自定义函数
            
            # # 4. 梯度范围检查
            # print("Gradient Value Ranges:")
            # print(f"  Num X: [{grad_numerical_x.min().item():.4e}, {grad_numerical_x.max().item():.4e}]")
            # print(f"  Ana X: [{grad_analytical_x.min().item():.4e}, {grad_analytical_x.max().item():.4e}]")
            # print(f"  Num Y: [{grad_numerical_y.min().item():.4e}, {grad_numerical_y.max().item():.4e}]")
            # print(f"  Ana Y: [{grad_analytical_y.min().item():.4e}, {grad_analytical_y.max().item():.4e}]")
            
            # 5. 移除异常值
            valid_mask = (
                torch.isfinite(grad_numerical_x) & 
                torch.isfinite(grad_analytical_x) &
                torch.isfinite(grad_numerical_y) & 
                torch.isfinite(grad_analytical_y)
            )
            
            # if not torch.all(valid_mask):
            #     invalid_count = torch.sum(~valid_mask).item()
            #     print(f"Warning: {invalid_count}/{valid_mask.numel()} invalid gradient points removed")
            
            # 6. 计算梯度误差 (添加稳定性)
            grad_diff_x = grad_numerical_x[valid_mask] - grad_analytical_x[valid_mask]
            grad_diff_y = grad_numerical_y[valid_mask] - grad_analytical_y[valid_mask]
            
            grad_error_sq = torch.mean(grad_diff_x**2 + grad_diff_y**2)
            grad_error_sq = torch.clamp(grad_error_sq, min=1e-16)  # 防止下溢
            
            # 7. 计算H1误差
            h1_error = torch.sqrt(l2_error_sq + grad_error_sq).item()
            
        except Exception as e:
            print(f"Error computing gradients: {str(e)}")
            h1_error = float('nan')
        
        # 打印结果
        print(f"Epoch [{epoch+1}/{epochs}], Total Loss: {loss.item():.4e}, "
            f"Boundary Loss: {boundary_loss.item():.4e}, Interior Loss: {interior_loss.item():.4e}, "
            f"Max Error: {max_error:.4e}, Average Error: {mean_error:.4e}, "
            f"L2 Error: {l2_error:.4e}, H1 Error: {h1_error:.4e}")

    # 自适应采样（可根据需要对L域进行调整）
    if (epoch + 1) % (epochs // num_adaptive_steps) == 0:
        numerical_solution = solver(X_interior, Y_interior)
        analytical_solution_values = analytical_solution(X_interior, Y_interior)
        error = torch.abs(numerical_solution - analytical_solution_values)

        _, high_error_indices = torch.topk(error.view(-1), num_high_error_samples)
        X_high_error, Y_high_error = X_interior[high_error_indices], Y_interior[high_error_indices]

        delta = 0.05
        X_new = X_high_error + (torch.rand_like(X_high_error) - 0.5) * delta
        Y_new = Y_high_error + (torch.rand_like(Y_high_error) - 0.5) * delta

        X_interior = torch.cat([X_interior.detach(), X_new.detach()], dim=0).requires_grad_(True)
        Y_interior = torch.cat([Y_interior.detach(), Y_new.detach()], dim=0).requires_grad_(True)
        num_high_error_samples += adaptive_sample_increment

# 生成损失图
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), loss_LK)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs (L-Shaped Domain)')
plt.show()

# 可视化数值解、解析解、误差
numerical_solution = solver(X, Y)
analytical_solution_values = analytical_solution(X, Y)
error = torch.abs(numerical_solution - analytical_solution_values)
max_error = torch.max(error).item()
print(f"Max Error: {max_error:.4e}")

plt.figure(figsize=(6, 5))  

# 误差图
plt.tricontourf(
    X.view(-1).detach().cpu().numpy(),
    Y.view(-1).detach().cpu().numpy(),
    error.detach().cpu().numpy().squeeze(),
    levels=40,
    cmap="inferno"
)
plt.colorbar()
plt.title("Error between Numerical and Analytical Solution")
plt.xlabel("x")
plt.ylabel("y")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from LegendreKANLayer import LegendreKANLayer
import random

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 设置随机种子
seed = 5
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# 定义 Monge-Ampère 方程的源项和解析解（针对圆域）
al = 5 / 3  # alpha 值

def source_function(x, y):
    r2 = x**2 + y**2
    return 4 * al**2 * (2 * al - 1) * r2**(2 * al - 2)

def analytical_solution(x, y):
    r2 = x**2 + y**2
    return r2**al

# 定义求解模型
class LegenKAN(nn.Module):
    def __init__(self):
        super(LegenKAN, self).__init__()
        self.legenkan1 = LegendreKANLayer(2, 8, 6)
        self.legenkan2 = LegendreKANLayer(8, 8, 6)
        self.legenkan3 = LegendreKANLayer(8, 1, 6)

    def forward(self, x, y):
        # 确保 x, y 为 [N,1]
        xy = torch.cat([x, y], dim=1)
        xy = self.legenkan1(xy)
        xy = self.legenkan2(xy)
        xy = self.legenkan3(xy)
        return xy

def analytical_gradient(x, y, al=5/3, eps=1e-12):
    """
    计算解析解 u = (x² + y²)^(3/4) 的梯度
    处理原点处的奇点问题
    
    参数:
    x, y: 输入张量
    al: 指数参数 (默认为3/4)
    eps: 避免除以零的小量
    
    返回:
    grad_x, grad_y: 对x和y的偏导数
    """
    # 计算半径平方
    r_sq = x**2 + y**2
    
    # 标识原点附近的点
    near_origin = r_sq < eps
    
    # 计算幂项 (al-1 = -1/4，需要避免除以零)
    safe_r_sq = torch.where(near_origin, eps, r_sq)
    power_term = al * torch.pow(safe_r_sq, al - 1)
    
    # 原点处设为零
    power_term = torch.where(near_origin, torch.zeros_like(power_term), power_term)
    
    # 计算梯度分量
    grad_x = power_term * (2 * x)
    grad_y = power_term * (2 * y)
    
    return grad_x, grad_y


# 将模型放到 GPU（或CPU）
solver = LegenKAN().to(device)

# 生成边界点：在圆周上均匀分布
num_boundary_points = 50000
theta = torch.linspace(0.0, 2.0*np.pi*(1 - 1/num_boundary_points), num_boundary_points, device=device)
X_boundary = torch.cos(theta).view(-1, 1)
Y_boundary = torch.sin(theta).view(-1, 1)

# 生成内部点：在[-1,1]^2中随机点，筛选出圆内点
num_interior_points = 400000
X_rand = (torch.rand(num_interior_points, 1, device=device)*2 - 1)
Y_rand = (torch.rand(num_interior_points, 1, device=device)*2 - 1)
r2 = X_rand**2 + Y_rand**2
inside_mask = r2 < 1.0

X_interior = X_rand[inside_mask].view(-1,1)
Y_interior = Y_rand[inside_mask].view(-1,1)
X_interior.requires_grad = True
Y_interior.requires_grad = True

# 损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(solver.parameters(), lr=0.01)

# 自适应采样参数
num_adaptive_steps = 5
num_high_error_samples = 200
learning_rate_decay = 0.8
adaptive_sample_increment = 20

loss_LK = []

epochs = 10000
alpha = 0.001
previous_loss = float('inf')
res = 200
x_plot = torch.linspace(-1, 1, res, device=device)
y_plot = torch.linspace(-1, 1, res, device=device)
X, Y = torch.meshgrid(x_plot, y_plot, indexing='ij')
X = X.reshape(-1, 1)
Y = Y.reshape(-1, 1)

# 仅保留圆内点
r2_global = X**2 + Y**2
inside_mask_global = r2_global <= 1.0
X = X[inside_mask_global].view(-1, 1)
Y = Y[inside_mask_global].view(-1, 1)
for epoch in range(epochs):
    optimizer.zero_grad()

    # 边界损失
    numerical_boundary = solver(X_boundary, Y_boundary)
    boundary_target = analytical_solution(X_boundary, Y_boundary)
    boundary_loss = criterion(numerical_boundary, boundary_target)

    # 内部点 Monge-Ampère损失
    numerical_interior = solver(X_interior, Y_interior)
    u_x = torch.autograd.grad(numerical_interior, X_interior, torch.ones_like(numerical_interior), create_graph=True)[0]
    u_y = torch.autograd.grad(numerical_interior, Y_interior, torch.ones_like(numerical_interior), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, X_interior, torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, Y_interior, torch.ones_like(u_y), create_graph=True)[0]
    u_xy = torch.autograd.grad(u_x, Y_interior, torch.ones_like(u_x), create_graph=True)[0]

    hessian_det = u_xx * u_yy - u_xy**2
    interior_loss = criterion(hessian_det, source_function(X_interior, Y_interior))

    # 总损失
    loss = boundary_loss + alpha * interior_loss

    # 反向传播与优化
    loss.backward(retain_graph=True)
    optimizer.step()

    loss_LK.append(loss.item())

    # 学习率调整
    if loss.item() > previous_loss:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= learning_rate_decay
        print(f"Epoch [{epoch+1}/{epochs}], Loss increased. Reducing learning rate to: {param_group['lr']:.6f}")
    previous_loss = loss.item()

    # 每200个epoch输出信息
    if (epoch + 1) % 200 == 0:
        # 确保输入需要梯度
        X.requires_grad_(True)
        Y.requires_grad_(True)
        
        # 重置梯度标志
        numerical_solution_global = solver(X, Y)
        analytical_solution_global = analytical_solution(X, Y)
        
        # 1. 计算函数值误差
        error_global = torch.abs(numerical_solution_global - analytical_solution_global)
        max_error = torch.max(error_global).item()
        mean_error = torch.mean(error_global).item()
        l2_error_sq = torch.mean((numerical_solution_global - analytical_solution_global)**2)
        l2_error = torch.sqrt(l2_error_sq).item()
        
        try:
            # 2. 计算数值解梯度
            grad_numerical_x, grad_numerical_y = torch.autograd.grad(
                outputs=numerical_solution_global,
                inputs=[X, Y],
                grad_outputs=torch.ones_like(numerical_solution_global),
                create_graph=False,
                retain_graph=True
            )
            
            # 3. 计算解析解梯度 (使用已知公式而非自动微分)
            grad_analytical_x, grad_analytical_y = analytical_gradient(X, Y)  # 推荐使用自定义函数
            
            # # 4. 梯度范围检查
            # print("Gradient Value Ranges:")
            # print(f"  Num X: [{grad_numerical_x.min().item():.4e}, {grad_numerical_x.max().item():.4e}]")
            # print(f"  Ana X: [{grad_analytical_x.min().item():.4e}, {grad_analytical_x.max().item():.4e}]")
            # print(f"  Num Y: [{grad_numerical_y.min().item():.4e}, {grad_numerical_y.max().item():.4e}]")
            # print(f"  Ana Y: [{grad_analytical_y.min().item():.4e}, {grad_analytical_y.max().item():.4e}]")
            
            # 5. 移除异常值
            valid_mask = (
                torch.isfinite(grad_numerical_x) & 
                torch.isfinite(grad_analytical_x) &
                torch.isfinite(grad_numerical_y) & 
                torch.isfinite(grad_analytical_y)
            )
            
            # if not torch.all(valid_mask):
            #     invalid_count = torch.sum(~valid_mask).item()
            #     print(f"Warning: {invalid_count}/{valid_mask.numel()} invalid gradient points removed")
            
            # 6. 计算梯度误差 (添加稳定性)
            grad_diff_x = grad_numerical_x[valid_mask] - grad_analytical_x[valid_mask]
            grad_diff_y = grad_numerical_y[valid_mask] - grad_analytical_y[valid_mask]
            
            grad_error_sq = torch.mean(grad_diff_x**2 + grad_diff_y**2)
            grad_error_sq = torch.clamp(grad_error_sq, min=1e-16)  # 防止下溢
            
            # 7. 计算H1误差
            h1_error = torch.sqrt(l2_error_sq + grad_error_sq).item()
            
        except Exception as e:
            print(f"Error computing gradients: {str(e)}")
            h1_error = float('nan')
        
        # 打印结果
        print(f"Epoch [{epoch+1}/{epochs}], Total Loss: {loss.item():.4e}, "
            f"Boundary Loss: {boundary_loss.item():.4e}, Interior Loss: {interior_loss.item():.4e}, "
            f"Max Error: {max_error:.4e}, Average Error: {mean_error:.4e}, "
            f"L2 Error: {l2_error:.4e}, H1 Error: {h1_error:.4e}")

    # 自适应采样
    if (epoch + 1) % (epochs // num_adaptive_steps) == 0:
        # 使用当前模型预测
        numerical_solution = solver(X_interior, Y_interior)

        # 计算一阶导数
        u_x = torch.autograd.grad(numerical_solution, X_interior, grad_outputs=torch.ones_like(numerical_solution), create_graph=True)[0]
        u_y = torch.autograd.grad(numerical_solution, Y_interior, grad_outputs=torch.ones_like(numerical_solution), create_graph=True)[0]

        # 计算二阶导数
        u_xx = torch.autograd.grad(u_x, X_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y, Y_interior, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
        u_xy = torch.autograd.grad(u_x, Y_interior, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

        # 计算残差（预测的 Hessian 行列式 - 源项）
        hessian_det = u_xx * u_yy - u_xy**2
        residual = torch.abs(hessian_det - source_function(X_interior, Y_interior))

        # 选取残差最大的点
        _, high_error_indices = torch.topk(residual.view(-1), num_high_error_samples)
        X_high_error, Y_high_error = X_interior[high_error_indices], Y_interior[high_error_indices]

        # 以这些点为中心采样新点
        delta = 0.05
        X_new = X_high_error + (torch.rand_like(X_high_error) - 0.5) * delta
        Y_new = Y_high_error + (torch.rand_like(Y_high_error) - 0.5) * delta

        # 限制在 [0, 1] 范围内，防止越界
        X_new = torch.clamp(X_new, -0.5, 0.5)
        Y_new = torch.clamp(Y_new, -0.5, 0.5)

        # 添加新点
        X_interior = torch.cat([X_interior, X_new.requires_grad_(True)])
        Y_interior = torch.cat([Y_interior, Y_new.requires_grad_(True)])
        num_high_error_samples += adaptive_sample_increment


# 生成损失图
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), loss_LK)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs (Circle Domain)')
plt.show()

# 可视化数值解、解析解、误差
res = 200
x_plot = torch.linspace(-1,1,res, device=device)
y_plot = torch.linspace(-1,1,res, device=device)
X_plot, Y_plot = torch.meshgrid(x_plot, y_plot, indexing='ij')
X_plot_flat = X_plot.reshape(-1,1)
Y_plot_flat = Y_plot.reshape(-1,1)

r2_plot = X_plot_flat**2 + Y_plot_flat**2
inside_plot = r2_plot <= 1.0

X_inside_plot = X_plot_flat[inside_plot].view(-1,1)
Y_inside_plot = Y_plot_flat[inside_plot].view(-1,1)

numerical_solution = solver(X_inside_plot, Y_inside_plot)
analytical_solution_values = analytical_solution(X_inside_plot, Y_inside_plot)
error = torch.abs(numerical_solution - analytical_solution_values)
max_error = torch.max(error).item()
print(f"Max Error: {max_error:.4e}")

X_inside_np = X_inside_plot.cpu().detach().numpy().ravel()
Y_inside_np = Y_inside_plot.cpu().detach().numpy().ravel()
num_sol_np = numerical_solution.cpu().detach().numpy().ravel()
ana_sol_np = analytical_solution_values.cpu().detach().numpy().ravel()
err_np = error.cpu().detach().numpy().ravel()

# 生成损失图
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs + 1), loss_LK)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs (L-Shaped Domain)')
plt.show()

# 可视化数值解、解析解、误差
numerical_solution = solver(X, Y)
analytical_solution_values = analytical_solution(X, Y)
error = torch.abs(numerical_solution - analytical_solution_values)
max_error = torch.max(error).item()
print(f"Max Error: {max_error:.4e}")

plt.figure(figsize=(6, 5))  

# 误差图
plt.tricontourf(
    X.view(-1).detach().cpu().numpy(),
    Y.view(-1).detach().cpu().numpy(),
    error.detach().cpu().numpy().squeeze(),
    levels=40,
    cmap="inferno"
)
plt.colorbar()
plt.title("Error between Numerical and Analytical Solution")
plt.xlabel("x")
plt.ylabel("y")

plt.tight_layout()
plt.show()